In [17]:
# https://www.ft.dk/-/media/sites/ft/pdf/dokumenter/aabne-data/oda-browser_brugervejledning.pdf
# https://oda.ft.dk/Home/OdataQuery

In [18]:
location = "ftp://oda.ft.dk/ODAXML/Referat/"

In [19]:
from ftplib import FTP
from pathlib import Path

HOST = "oda.ft.dk"
ROOT = "/ODAXML/Referat"


def connect():
    ftp = FTP(HOST, timeout=60)
    ftp.login()  # anonymous
    return ftp


def ls(path):
    ftp = connect()
    ftp.cwd(path)
    items = sorted(ftp.nlst())
    ftp.quit()
    return items


# top-level content
ls(ROOT)

['Dokumentation', 'Skemaer', 'samling']

In [20]:
# samlinger = parliamentary sessions (e.g. 20231 = 2023/24 first session)
samlinger = ls(f"{ROOT}/samling")
print(len(samlinger), "sessions")
samlinger

22 sessions


['20091',
 '20101',
 '20102',
 '20111',
 '20121',
 '20131',
 '20141',
 '20142',
 '20151',
 '20161',
 '20171',
 '20181',
 '20182',
 '20191',
 '20201',
 '20211',
 '20221',
 '20222',
 '20231',
 '20241',
 '20251',
 '20252']

In [21]:
# meetings in one session: files named {samling}_M{n}_helemoedet.xml (= whole meeting)
def meetings(samling):
    return ls(f"{ROOT}/samling/{samling}")


meetings("20231")[:10]

['20231_M100_helemoedet.xml',
 '20231_M101_helemoedet.xml',
 '20231_M102_helemoedet.xml',
 '20231_M103_helemoedet.xml',
 '20231_M104_helemoedet.xml',
 '20231_M105_helemoedet.xml',
 '20231_M106_helemoedet.xml',
 '20231_M10_helemoedet.xml',
 '20231_M11_helemoedet.xml',
 '20231_M12_helemoedet.xml']

In [22]:
# download one meeting XML to local data dir
OUT = Path("../data/ft/referat")
OUT.mkdir(parents=True, exist_ok=True)


def download(samling, fname, out_dir=OUT):
    dest = Path(out_dir) / fname
    ftp = connect()
    with open(dest, "wb") as f:
        ftp.retrbinary(f"RETR {ROOT}/samling/{samling}/{fname}", f.write)
    ftp.quit()
    return dest


sample = download("20231", "20231_M1_helemoedet.xml")
print(sample, sample.stat().st_size, "bytes")

../data/ft/referat/20231_M1_helemoedet.xml 74148 bytes


In [23]:
# peek: parse XML, show structure + first bits of speech text
import xml.etree.ElementTree as ET

tree = ET.parse(sample)
root = tree.getroot()


def localname(tag):
    return tag.split("}")[-1]


# tag frequency
from collections import Counter

tags = Counter(localname(e.tag) for e in root.iter())
print(tags.most_common(15))

# first non-empty text chunks
texts = [e.text.strip() for e in root.iter() if e.text and e.text.strip()]
for t in texts[:15]:
    print("-", t[:120])

[('Char', 515), ('Linea', 497), ('Exitus', 122), ('TekstGruppe', 30), ('TaleSegment', 19), ('MetaSpeechSegment', 19), ('LastModified', 19), ('EdixiStatus', 19), ('StartDateTime', 19), ('EndDateTime', 18), ('PunktTekst', 11), ('Tale', 11), ('Taler', 11), ('MetaSpeakerMP', 11), ('OratorFirstName', 11)]
- 20231
- Folketinget
- 1
- 2023-10-03T12:00:00
- Folketingssalen
- W:\20231\M001\
- X:\Arbor\FT2023\001\SALEN\
- 1. møde
- Tirsdag den 3. oktober 2023 kl. 12.00
- Dagsorden
- 1) Valg af formand.
- 2) Valg af 4 næstformænd.
- 3) Valg af 4 tingsekretærer.
- 4) Statsministerens redegørelse i henhold til grundlovens § 38.
- 0


## XML structure (LexDania 2.1)

```
Dokument
├─ MetaMeeting            ParliamentarySession, MeetingNumber, DateOfSitting, Location
└─ DagsordenPunkt*        (agenda item)
   ├─ MetaFTAgendaItem    ShortTitle, FTCaseType, FTCaseNumber
   └─ Aktivitet
      └─ Tale*            (one speaker turn = one statement)
         ├─ Taler/MetaSpeakerMP   OratorFirstName, OratorLastName, GroupNameShort, OratorRole
         └─ TaleSegment*  (statement may be split into segments)
            ├─ MetaSpeechSegment   StartDateTime, EndDateTime
            └─ TekstGruppe/.../Linea/Char   (the spoken text)
```

Row = one `Tale`. Timestamps come from its `TaleSegment`s (start = first segment start, end = last segment end).


In [24]:
# parse a meeting XML -> list of statement rows (one row per Tale)
def ln(tag):
    return tag.split("}")[-1]


def descendants(elem, name):
    return [e for e in elem.iter() if ln(e.tag) == name]


def text_of(elem):
    """Join spoken text under an element. Linea = line, Char = text run."""
    lines = []
    for linea in descendants(elem, "Linea"):
        chars = "".join(c.text or "" for c in descendants(linea, "Char"))
        if chars.strip():
            lines.append(chars.strip())
    return "\n".join(lines)


def txt(elem, name):
    n = descendants(elem, name)
    return n[0].text.strip() if n and n[0].text else ""


COLUMNS = ["session", "meeting", "date", "agenda_item", "case_no",
           "start", "end", "speaker", "group", "role", "text"]


def parse_meeting(path):
    root = ET.parse(path).getroot()
    meta = descendants(root, "MetaMeeting")[0]
    session = txt(meta, "ParliamentarySession")
    meeting_no = txt(meta, "MeetingNumber")
    date = txt(meta, "DateOfSitting")

    rows = []
    for item in descendants(root, "DagsordenPunkt"):
        agenda = txt(item, "ShortTitle")
        case_no = txt(item, "FTCaseNumber")
        for tale in descendants(item, "Tale"):
            sp = (descendants(tale, "MetaSpeakerMP") or [None])[0]
            first = txt(sp, "OratorFirstName") if sp is not None else ""
            last = txt(sp, "OratorLastName") if sp is not None else ""
            group = txt(sp, "GroupNameShort") if sp is not None else ""
            role = txt(sp, "OratorRole") if sp is not None else ""

            segs = descendants(tale, "TaleSegment")
            starts = [txt(s, "StartDateTime") for s in segs if txt(s, "StartDateTime")]
            ends = [txt(s, "EndDateTime") for s in segs if txt(s, "EndDateTime")]
            body = "\n".join(text_of(s) for s in segs).strip()
            if not body:
                continue
            rows.append({
                "session": session,
                "meeting": meeting_no,
                "date": date,
                "agenda_item": agenda,
                "case_no": case_no,
                "start": min(starts) if starts else "",
                "end": max(ends) if ends else "",
                "speaker": f"{first} {last}".strip(),
                "group": group,
                "role": role,
                "text": body,
            })
    return rows


rows = parse_meeting(sample)
print(len(rows), "statements")
for r in rows[:8]:
    print(f"{r['start'][11:]}-{r['end'][11:]} | {r['speaker']:<22} | {r['group']:<4} | "
          f"{r['role']:<16} | {r['text'][:60]!r}")

11 statements
12:00:46-12:03:28 | Pia Kjærsgaard         | DF   | aldersformanden  | 'Deres Majestæt og Kongelige Højheder.\nFolketingets åbning er'
12:03:28-12:03:41 | Pia Kjærsgaard         | DF   | aldersformanden  | 'Til formand har samtlige partier indstillet hr.\nSøren Gade.\n'
12:03:41-12:04:21 | Pia Kjærsgaard         | DF   | aldersformanden  | 'Til første næstformand har Socialdemokratiets gruppe udpeget'
12:04:21-12:05:12 | Pia Kjærsgaard         | DF   | aldersformanden  | 'Til tingsekretærer er i henhold til de indgåede valgforbund '
12:05:12-12:08:05 | Søren Gade             | V    | formand          | 'Tak.\nNæste gang jeg fylder rundt, bliver jeg også 70 år, så '
12:08:05-12:08:44 | Søren Gade             | V    | formand          | 'Så er det jo demokratiets festdag, en fest, vi sammen holder'
12:08:44-12:09:02 | Søren Gade             | V    | formand          | 'Herefter vil jeg give ordet til statsministeren, så statsmin'
12:09:02-12:52:05 | Mette Frederiksen      |

In [25]:
# export to CSV (stdlib csv, no pandas dependency)
import csv

CSV_OUT = Path("../data/ft/csv")
CSV_OUT.mkdir(parents=True, exist_ok=True)


def write_csv(rows, dest):
    with open(dest, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=COLUMNS)
        w.writeheader()
        w.writerows(rows)
    return dest


out = write_csv(rows, CSV_OUT / (sample.stem + ".csv"))
print("wrote", out, "-", len(rows), "rows")

wrote ../data/ft/csv/20231_M1_helemoedet.csv - 11 rows


In [26]:
# batch: download + parse EVERY meeting in EVERY samling -> one CSV per session + combined
def process_meeting(samling, fname, ftp):
    """Download (if missing) and parse one meeting, reusing an open FTP conn."""
    dest = OUT / fname
    if not dest.exists():
        with open(dest, "wb") as f:
            ftp.retrbinary(f"RETR {ROOT}/samling/{samling}/{fname}", f.write)
    try:
        return parse_meeting(dest)
    except ET.ParseError as e:
        print("  ! parse failed", fname, e)
        return []


def process_all(sessions=None, combined_name="ft_referater_all.csv"):
    sessions = sessions or ls(f"{ROOT}/samling")
    combined = CSV_OUT / combined_name
    total = 0
    with open(combined, "w", newline="", encoding="utf-8") as cf:
        cw = csv.DictWriter(cf, fieldnames=COLUMNS)
        cw.writeheader()
        for s in sessions:
            files = [f for f in ls(f"{ROOT}/samling/{s}") if f.endswith(".xml")]
            ftp = connect()
            session_rows = []
            for fname in files:
                rows = process_meeting(s, fname, ftp)
                session_rows.extend(rows)
                cw.writerows(rows)
            ftp.quit()
            write_csv(session_rows, CSV_OUT / f"{s}.csv")
            total += len(session_rows)
            print(f"{s}: {len(files):>3} meetings -> {len(session_rows):>6} statements")
    print(f"\nDONE: {total} statements -> {combined}")
    return combined


# run for all sessions (downloads ~1000s of files; slow on first run)
process_all()

20091: 109 meetings ->  61299 statements
20101: 108 meetings ->  41518 statements
20102:   2 meetings ->     18 statements
20111: 102 meetings ->  49148 statements
20121: 111 meetings ->  42102 statements
20131: 109 meetings ->  38520 statements
20141:  98 meetings ->  36235 statements
20142:   6 meetings ->   1210 statements
20151: 112 meetings ->  58758 statements
20161: 113 meetings ->  55124 statements
20171: 108 meetings ->  41133 statements
20181:  93 meetings ->  38055 statements
20182:   8 meetings ->     91 statements
20191: 151 meetings ->  57782 statements
20201: 138 meetings ->  67725 statements
20211: 135 meetings ->  46899 statements
20221:   3 meetings ->   1622 statements
  ! parse failed 20222_M62_helemoedet.xml not well-formed (invalid token): line 3, column 435099
20222:  73 meetings ->  39793 statements
20231: 106 meetings ->  57513 statements
20241: 112 meetings ->  58019 statements
20251:  57 meetings ->  24286 statements
20252:  12 meetings ->   3319 statements



PosixPath('../data/ft/csv/ft_referater_all.csv')